# Topic 7a - LoRA from Scratch

Barclays Customer Support Intelligence System | Topic 7a

## What you will build

Full fine-tuning updates every weight in the model: millions of parameters.
LoRA freezes all weights and injects two tiny matrices per layer whose product
approximates the weight change. You implement LoRA from scratch in PyTorch,
apply it to a simple classifier to verify the math, then use HuggingFace PEFT
to LoRA-fine-tune Flan-T5-small on complaint summarization in a GPU training job.

## Why this matters to Barclays

Barclays hosts dozens of NLP models for complaint triage, summarization, and
risk flagging. Full fine-tuning for each task is expensive: storage, GPU time,
and deployment overhead multiply with every new model version.
LoRA adapters are a few MB each. You deploy one frozen base model and swap adapters
per task: the same architecture that powers production PEFT deployments at scale.

## Learning objectives

1. Derive the LoRA update rule: W' = W + B @ A, explain why B=zeros at init
2. Implement LoraLayer in PyTorch, freeze the original layer, verify gradient flow
3. Swap linear layers in an FFN for LoraLayer, count trainable parameters before and after
4. Explain rank r and alpha and how to choose them (r=4,8,16 heuristics)
5. Use HuggingFace PEFT (LoraConfig, get_peft_model) to apply LoRA to Flan-T5
6. Launch and monitor a LoRA fine-tuning job on SageMaker GPU (ml.g4dn.xlarge)

## Estimated time

90 to 120 minutes in class.


In [ ]:
# Environment setup for SageMaker Studio.
# Section 1-3 (scratch LoRA) runs in this CPU kernel.
# Section 4 (Flan-T5 capstone) launches a remote GPU job via HuggingFace estimator.

!pip install -q "sagemaker>=2.200.0,<3.0.0" \
    "transformers>=4.35.0,<4.40.0" \
    "tokenizers>=0.15.0,<0.20.0" \
    "numpy<2" \
    "matplotlib>=3.7.0"

import sagemaker
from sagemaker import get_execution_role
import boto3

sess   = sagemaker.Session()
role   = get_execution_role()
bucket = sess.default_bucket()
region = sess.boto_region_name

print(f"Role:   {role}")
print(f"Bucket: {bucket}")
print(f"Region: {region}")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import os
import random
import warnings

warnings.filterwarnings("ignore")

def set_seeds(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)

# CPU for all in-notebook demos; GPU job runs in scripts_topic7a/train.py.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")


## Section 1: The Parameter Budget Problem

Every weight matrix W in a model stores d x k numbers.
Fine-tuning changes every single one. For a single 512 x 1024 layer that is 524,288
numbers. For Flan-T5-small (250M parameters) you update every one of them.

What if you only needed to store the *change* in W instead of the full update?
The change is called delta_W. Full fine-tuning stores delta_W as a dense d x k matrix.
LoRA asks: can we approximate delta_W with a low-rank factorisation delta_W = B @ A?
If r is much smaller than min(d, k) the saving is enormous.

First, let's feel the pain of storing full weight deltas.


In [ ]:
# Beat 1 -- try to store a full weight delta for every layer in an FFN.
# This shows WHY we need a smarter approach before introducing LoRA.

# A realistic small FFN: six layers with dimensions matching our demo model.
layer_shapes = [
    (784, 1024),
    (1024, 512),
    (512, 256),
    (256, 128),
    (128, 64),
    (64, 10),
]

print("Full weight delta storage (one float32 per parameter):")
print("-" * 55)
total_delta_params = 0
for i, (d_in, d_out) in enumerate(layer_shapes, 1):
    n_params = d_in * d_out
    mb = n_params * 4 / (1024 ** 2)
    total_delta_params += n_params
    print(f"  fc{i}: {d_in} x {d_out} = {n_params:>9,} params | {mb:.2f} MB delta")

total_mb = total_delta_params * 4 / (1024 ** 2)
print("-" * 55)
print(f"  Total: {total_delta_params:,} params | {total_mb:.2f} MB for ONE fine-tuned copy")
print()
print("Scale this to Flan-T5-small (250M params):")
flant5_delta_mb = 250_000_000 * 4 / (1024 ** 2)
print(f"  Full delta: {flant5_delta_mb:.0f} MB just for the weight changes")
print()
print("And you need a separate copy for EVERY downstream task.")
print("Complaint summarization: +950 MB. NER: +950 MB. Risk: +950 MB ...")
print("This does not scale. We need a better approach.")


In [ ]:
# Beat 1b -- what if we try to 'compress' delta_W but set rank=d (full rank)?
# Spoiler: r=d gives ZERO compression. This is why rank selection matters.

print("LoRA parameter count as a function of rank r:")
print("Layer fc1: d=784, k=1024")
d, k = 784, 1024
print(f"  Full fine-tuning delta: d * k = {d * k:,} params")
print()
for r in [d, 256, 64, 16, 8, 4, 1]:
    lora_params = d * r + r * k
    ratio = (d * k) / lora_params
    note = " <- same as full fine-tuning!" if r >= min(d, k) else ""
    print(f"  r={r:>4}: A({r}x{k}) + B({d}x{r}) = {lora_params:>9,} params | "
          f"{ratio:>6.1f}x compression{note}")

print()
print("Takeaway: r must be much smaller than d and k to get meaningful compression.")
print("r=8 gives ~58x compression on fc1. That is the insight behind LoRA.")


## The LoRA Idea

Instead of updating W directly, we inject two small matrices:

    W' = W + B @ A

where W stays frozen (no gradient), and only A and B are trained.

- A has shape  (r, k):  projects from input dimension k down to rank r
- B has shape  (d, r):  projects from rank r up to output dimension d
- r much smaller than min(d, k) ensures we store far fewer parameters

At initialisation: A ~ Normal(0, 0.02), B = 0.
This means B @ A = 0 at step 0, so the adapted model starts identical to the
pre-trained model. Only during training do A and B diverge from this baseline.

The scaling factor lora_alpha / r controls how much the LoRA update contributes
relative to the frozen weight. A common heuristic is alpha = 2 * r.

<!-- DIAGRAM: LoRA low-rank decomposition -->
[View diagram](../../plans/topic_7a_lora_ffn/diagrams/lora-decomposition.mmd)


In [ ]:
# Beat 3 -- implement LoraLayer in PyTorch.
# This is the core of the entire topic. Read every comment carefully.
# The PEFT library does exactly this internally -- but you need to understand it
# before you trust a library to do it for you.

class LoraLayer(nn.Module):
    """
    Wraps an existing nn.Linear with LoRA adaptation.

    The forward pass computes:
        output = original_layer(x) + (lora_B @ lora_A)(x) * scale

    where scale = lora_alpha / rank, lora_A is (rank, in_features),
    lora_B is (out_features, rank).

    At initialisation lora_B is all zeros, so the adaptation contributes
    nothing at step 0. Only lora_A and lora_B are trainable; original_layer
    parameters are frozen.
    """

    def __init__(self, original_layer: nn.Linear, rank: int = 8, lora_alpha: int = 16):
        super().__init__()

        self.in_features  = original_layer.in_features
        self.out_features = original_layer.out_features
        self.rank         = rank
        self.scale        = lora_alpha / rank

        # Freeze the original layer
        self.original_layer = original_layer
        for param in self.original_layer.parameters():
            param.requires_grad = False

        # LoRA low-rank matrices
        # lora_A: (rank, in_features) -- "down" projection
        # lora_B: (out_features, rank) -- "up" projection
        self.lora_A = nn.Linear(self.in_features, rank, bias=False)
        self.lora_B = nn.Linear(rank, self.out_features, bias=False)

        # Initialisation: A ~ Normal(0, 0.02), B = 0.
        # At step 0: lora_B(lora_A(x)) = 0 for all x.
        nn.init.normal_(self.lora_A.weight, std=0.02)
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Frozen path: no gradient computation
        original_output = self.original_layer(x)

        # LoRA path: lora_B(lora_A(x)) * scale
        lora_output = self.lora_B(self.lora_A(x)) * self.scale

        return original_output + lora_output

    def merge_weights(self) -> nn.Linear:
        """
        Merge A and B into the original weight for zero-overhead inference.
        After merging you can discard lora_A and lora_B entirely.
        """
        merged = nn.Linear(self.in_features, self.out_features,
                           bias=self.original_layer.bias is not None)
        with torch.no_grad():
            delta_W = self.lora_B.weight @ self.lora_A.weight
            merged.weight.copy_(self.original_layer.weight + delta_W * self.scale)
            if self.original_layer.bias is not None:
                merged.bias.copy_(self.original_layer.bias)
        return merged


# Quick sanity check
test_linear = nn.Linear(64, 128)
lora_test   = LoraLayer(test_linear, rank=4, lora_alpha=8)

x_test = torch.randn(16, 64)
out    = lora_test(x_test)

print("LoraLayer sanity check:")
print(f"  Input shape:  {x_test.shape}")
print(f"  Output shape: {out.shape}")
print(f"  original_layer grad: {lora_test.original_layer.weight.requires_grad}")
print(f"  lora_A grad:         {lora_test.lora_A.weight.requires_grad}")
print(f"  lora_B grad:         {lora_test.lora_B.weight.requires_grad}")
print(f"  lora_B weight sum (should be 0.0): {lora_test.lora_B.weight.sum().item():.4f}")
print()
merged = lora_test.merge_weights()
print(f"  merge_weights output shape: {merged.weight.shape}  (no lora overhead at inference)")


## Lab 1: Implement the LoraLayer (Tier 1 Guided, 15 min)

### Situation

Your Barclays team has a pre-trained complaint classifier. You want to adapt it for a
new sub-task (priority routing) without retraining the whole model.
The team lead says: "Implement LoRA so we can inject trainable adapters without
touching the frozen backbone."

### Task

Complete the `LoraLayerStudent` class below. The class wraps an existing nn.Linear
and adds low-rank adaptation.

### Action

Follow the numbered steps in the comments. Do not change any line that is already filled in.

### Result

The verification cell at the bottom must print:
- original_layer.requires_grad = False
- lora_A.requires_grad = True, lora_B.requires_grad = True
- lora_B initial weight sum = 0.0
- Output shape correct: torch.Size([8, 32])
- merge_weights runs without error


In [ ]:
class LoraLayerStudent(nn.Module):
    def __init__(self, original_layer: nn.Linear, rank: int = 8, lora_alpha: int = 16):
        super().__init__()
        self.in_features  = original_layer.in_features
        self.out_features = original_layer.out_features
        self.rank         = rank
        self.scale        = lora_alpha / rank

        # Step 1: Store original_layer and freeze its parameters.
        self.original_layer = original_layer
        for param in self.original_layer.parameters():
            param.requires_grad = False

        # Step 2: Create self.lora_A as a Linear layer from in_features to rank with no bias.
        self.lora_A = nn.Linear(self.in_features, rank, bias=False)

        # Step 3: Create self.lora_B as a Linear layer from rank to out_features with no bias.
        self.lora_B = nn.Linear(rank, self.out_features, bias=False)

        # Step 4: Initialise lora_A.weight with Normal(0, 0.02) and lora_B.weight with zeros.
        nn.init.normal_(self.lora_A.weight, std=0.02)
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Step 5: Compute the original (frozen) output.
        original_output = self.original_layer(x)

        # Step 6: Compute the LoRA output scaled by self.scale.
        lora_output = self.lora_B(self.lora_A(x)) * self.scale

        # Step 7: Return the sum of the two outputs.
        return original_output + lora_output


In [ ]:
# Safety-net: run this if you did not finish Lab 1.
# SKIP this cell if you DID finish Lab 1.
_needs_safety = False
try:
    _probe = LoraLayerStudent(nn.Linear(4, 8))
    if _probe.lora_A is None or _probe.lora_B is None or _probe.original_layer is None:
        _needs_safety = True
except Exception:
    _needs_safety = True

if _needs_safety:
    print("Using Lab 1 safety-net so the rest of the notebook can run.")
    LoraLayerStudent = LoraLayer
else:
    print("Lab 1 complete: using your LoraLayerStudent.")


In [ ]:
# Verification: do not edit this cell.
test_lin = nn.Linear(16, 32)
student_lora = LoraLayerStudent(test_lin, rank=4, lora_alpha=8)

x_v = torch.randn(8, 16)
out_v = student_lora(x_v)

assert student_lora.original_layer.weight.requires_grad == False, \
    "original_layer must be frozen (requires_grad=False)"
assert student_lora.lora_A.weight.requires_grad == True, \
    "lora_A must be trainable (requires_grad=True)"
assert student_lora.lora_B.weight.requires_grad == True, \
    "lora_B must be trainable (requires_grad=True)"
assert abs(student_lora.lora_B.weight.sum().item()) < 1e-6, \
    "lora_B must be initialised to zeros"
assert out_v.shape == torch.Size([8, 32]), \
    f"Output shape wrong: got {out_v.shape}, expected (8, 32)"

print("original_layer.requires_grad =", student_lora.original_layer.weight.requires_grad)
print("lora_A.requires_grad =", student_lora.lora_A.weight.requires_grad)
print("lora_B.requires_grad =", student_lora.lora_B.weight.requires_grad)
print("lora_B initial weight sum =", round(student_lora.lora_B.weight.sum().item(), 4))
print("Output shape:", out_v.shape)
print()
print("Lab 1 verification PASSED.")


### Stretch (fast finishers)

Implement `merge_weights(self) -> nn.Linear` on your `LoraLayerStudent`.
The merged weight matrix should equal `W + scale * (B.weight @ A.weight)`.
Verify that `merged_layer(x)` and `lora_layer(x)` produce identical outputs
(the difference should be less than 1e-5 due to float32 precision).

### Homework Extension

Try different initialisations for lora_A and lora_B:
- lora_A = zeros, lora_B = Normal(0, 0.02): what happens to the output at step 0?
- Both = Normal(0, 0.02): does training still converge? Why or why not?

Write a short explanation (2-3 sentences) of why the standard init (A normal, B zero)
is preferred and what property it preserves at the start of training.


In [ ]:
# Homework Extension: try alternative initialisations and compare initial outputs.

alt_lora = LoraLayer(nn.Linear(16, 32), rank=4, lora_alpha=8)

# Alternative init: A zeros, B normal -- output is still zero at step 0 (symmetric to standard init).
nn.init.zeros_(alt_lora.lora_A.weight)
nn.init.normal_(alt_lora.lora_B.weight, std=0.02)

x_hw = torch.randn(8, 16)
lora_contribution = alt_lora.lora_B(alt_lora.lora_A(x_hw)) * alt_lora.scale
print("LoRA contribution norm (A=0, B~Normal):", lora_contribution.norm().item())
print("Initial total output norm:", alt_lora(x_hw).norm().item())

# Symmetric init: both Normal -- output is non-zero at step 0, base model is perturbed immediately.
alt2 = LoraLayer(nn.Linear(16, 32), rank=4, lora_alpha=8)
nn.init.normal_(alt2.lora_A.weight, std=0.02)
nn.init.normal_(alt2.lora_B.weight, std=0.02)
lora_contribution2 = alt2.lora_B(alt2.lora_A(x_hw)) * alt2.scale
print("LoRA contribution norm (both Normal):", lora_contribution2.norm().item())
print("Standard init (A normal, B zero) is preferred because it preserves the pre-trained")
print("model exactly at step 0; training then learns the delta from zero.")


### Peer Discussion (3 min): Choosing LoRA for Barclays

Your team is evaluating LoRA for Barclays' complaint-routing model. Before you swap any layers, discuss:

1. What rank `r` would you choose for a model that handles eight complaint sub-categories, and why? What are the tradeoffs between expressiveness (higher r) and parameter count (lower r)?
2. Which weight matrices would you target first: the attention Q/K/V projections, or the FFN linear layers? What is the consequence if you pick the wrong set?
3. A colleague argues: "Just fine-tune the full model end to end. Storage is cheap, and we already have the GPU budget." How do you respond, given that Barclays runs five different downstream tasks on the same base model?
4. What if your validation loss plateaus early at rank=4? Do you increase rank, increase `lora_alpha`, or change the target modules first? Justify your order of experiments.


## Section 2: Applying LoRA to a Feed-Forward Network

We now apply your LoraLayer to a real (small) FFN pre-trained on FashionMNIST.
The goal is to adapt it to MNIST with only the LoRA parameters trainable.

This mirrors what we will do with Flan-T5 in Section 4: the FFN version
is just faster and requires no GPU, so we can verify the entire workflow here.

Before we automate anything, let's feel the pain of the naive approach: manually
inspecting each linear layer one by one to figure out what to swap. This is what
LoRA's `replace_fc_with_lora` helper saves us from.


In [ ]:
# Beat 1 -- naive approach: manually inspect each layer to plan what to wrap.
# This is tedious and error-prone. Imagine doing this for a 250M parameter model
# with hundreds of linear layers. The automation we build next (replace_fc_with_lora)
# is what saves us from this.

# A throwaway preview model with the same architecture we will pre-train below.
_preview = nn.Sequential(
    nn.Linear(784, 1024),
    nn.Linear(1024, 512),
    nn.Linear(512, 256),
    nn.Linear(256, 128),
    nn.Linear(128, 64),
    nn.Linear(64, 10),
)

print("Naive layer-by-layer inspection (the pain LoRA tooling removes):")
print("-" * 60)
total_params = 0
for idx, layer in enumerate(_preview, start=1):
    n = layer.in_features * layer.out_features
    total_params += n
    candidate = "wrap?" if n > 50_000 else "skip "
    print(f"  fc{idx}: in={layer.in_features:>4} out={layer.out_features:>4} "
          f"params={n:>9,} | manual decision: {candidate}")
print("-" * 60)
print(f"  Total params to consider: {total_params:,}")
print()
print("Problems with this naive process:")
print("  1. Easy to miss a layer or wrap the wrong one")
print("  2. Does not scale to models with hundreds of linear layers")
print("  3. No consistent rank/alpha policy across layers")
print("  4. No way to freeze the rest of the model in the same pass")
print()
print("Next: a single helper replace_fc_with_lora that does all of this for us.")

del _preview


Parameter comparison diagram below shows what happens to trainable parameter
count when we swap linear layers for LoRA-wrapped ones.

<!-- DIAGRAM: LoRA parameter count comparison -->
[View diagram](../../plans/topic_7a_lora_ffn/diagrams/lora-parameter-comparison.mmd)


In [ ]:
# Pre-train a simple FFN on FashionMNIST.
# We use this as our "pre-trained model" that we will later adapt with LoRA.
# Training takes ~3 minutes on CPU; the accuracy will be ~87% on FashionMNIST.

from tqdm.auto import tqdm

transform = transforms.Compose([transforms.ToTensor()])
fashion_train = datasets.FashionMNIST(root="./data", train=True,  download=True, transform=transform)
fashion_test  = datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)
fashion_train_loader = DataLoader(fashion_train, batch_size=128, shuffle=True)
fashion_test_loader  = DataLoader(fashion_test,  batch_size=128, shuffle=False)


class FFNModel(nn.Module):
    """
    Five-layer feed-forward network with BatchNorm and Dropout.
    Trained on FashionMNIST (10 classes, 28x28 images).
    We will later replace fc1-fc3 with LoraLayer wrappers.
    """
    def __init__(self):
        super().__init__()
        self.flatten  = nn.Flatten()
        self.fc1      = nn.Linear(784, 1024)
        self.bn1      = nn.BatchNorm1d(1024)
        self.drop1    = nn.Dropout(0.3)
        self.fc2      = nn.Linear(1024, 512)
        self.bn2      = nn.BatchNorm1d(512)
        self.drop2    = nn.Dropout(0.3)
        self.fc3      = nn.Linear(512, 256)
        self.bn3      = nn.BatchNorm1d(256)
        self.drop3    = nn.Dropout(0.3)
        self.fc4      = nn.Linear(256, 128)
        self.bn4      = nn.BatchNorm1d(128)
        self.drop4    = nn.Dropout(0.3)
        self.fc5      = nn.Linear(128, 64)
        self.bn5      = nn.BatchNorm1d(64)
        self.drop5    = nn.Dropout(0.3)
        self.fc6      = nn.Linear(64, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.drop1(self.bn1(F.relu(self.fc1(x))))
        x = self.drop2(self.bn2(F.relu(self.fc2(x))))
        x = self.drop3(self.bn3(F.relu(self.fc3(x))))
        x = self.drop4(self.bn4(F.relu(self.fc4(x))))
        x = self.drop5(self.bn5(F.relu(self.fc5(x))))
        return self.fc6(x)


pretrained_model = FFNModel().to(device)
optimizer_pre    = torch.optim.Adam(pretrained_model.parameters(), lr=1e-3)
criterion        = nn.CrossEntropyLoss()

PRETRAIN_EPOCHS = 5
for epoch in range(PRETRAIN_EPOCHS):
    pretrained_model.train()
    total_loss = 0
    for images, labels in tqdm(fashion_train_loader, desc=f"Pre-train {epoch+1}/{PRETRAIN_EPOCHS}", leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer_pre.zero_grad()
        loss = criterion(pretrained_model(images), labels)
        loss.backward()
        optimizer_pre.step()
        total_loss += loss.item()
    print(f"  Epoch {epoch+1}: avg loss = {total_loss / len(fashion_train_loader):.4f}")

total_pre  = sum(p.numel() for p in pretrained_model.parameters())
train_pre  = sum(p.numel() for p in pretrained_model.parameters() if p.requires_grad)
print(f"\nPre-trained model: {total_pre:,} total params, {train_pre:,} trainable")


In [ ]:
# Beat 3 -- replace fc1, fc2, fc3 with LoraLayer wrappers.
# fc4, fc5, fc6 stay frozen (not wrapped).
# Only the LoRA A and B matrices are trainable after replacement.

def replace_fc_with_lora(model: FFNModel, rank: int = 8, lora_alpha: int = 16) -> FFNModel:
    """
    Replace the first three fully-connected layers with LoraLayer.
    Deterministic: always wraps fc1, fc2, fc3.
    """
    model.fc1 = LoraLayer(model.fc1, rank=rank, lora_alpha=lora_alpha)
    model.fc2 = LoraLayer(model.fc2, rank=rank, lora_alpha=lora_alpha)
    model.fc3 = LoraLayer(model.fc3, rank=rank, lora_alpha=lora_alpha)
    # fc4, fc5, fc6 are left unchanged and frozen by not wrapping
    for param in model.fc4.parameters():
        param.requires_grad = False
    for param in model.fc5.parameters():
        param.requires_grad = False
    for param in model.fc6.parameters():
        param.requires_grad = False
    return model


lora_model = replace_fc_with_lora(pretrained_model, rank=8, lora_alpha=16)
lora_model.to(device)

total_lora    = sum(p.numel() for p in lora_model.parameters())
trainable_lora = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)

print("Parameter counts after LoRA injection:")
print(f"  Total params (frozen + LoRA): {total_lora:,}")
print(f"  Trainable (LoRA A+B only):    {trainable_lora:,}")
print(f"  Frozen:                       {total_lora - trainable_lora:,}")
print(f"  Compression ratio vs full FT: {train_pre / trainable_lora:.1f}x fewer trainable params")

lora_optimizer = torch.optim.Adam(
    [p for p in lora_model.parameters() if p.requires_grad], lr=3e-4)


In [ ]:
# Safety-net: run this if the previous cell failed or you skipped it.
# SKIP if lora_model and trainable_lora are already defined.
if 'lora_model' not in dir() or lora_model is None:
    print("Rebuilding lora_model from safety-net...")
    import copy
    _base = copy.deepcopy(pretrained_model) if 'pretrained_model' in dir() else FFNModel().to(device)
    lora_model = replace_fc_with_lora(_base, rank=8, lora_alpha=16)
    lora_model.to(device)
    total_lora    = sum(p.numel() for p in lora_model.parameters())
    trainable_lora = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
    if 'train_pre' not in dir():
        train_pre = sum(p.numel() for p in _base.parameters() if p.requires_grad)
    lora_optimizer = torch.optim.Adam(
        [p for p in lora_model.parameters() if p.requires_grad], lr=3e-4)
    print(f"lora_model ready. Trainable params: {trainable_lora:,}")
else:
    print("lora_model already defined, skipping safety-net.")


### Peer Discussion (3 min): Rank Trade-offs

You just saw the compression ratio for rank=8. Consider:

1. What happens to model quality if you set rank=1? rank=128?
2. In production, Barclays needs to serve five complaint sub-tasks from one base model.
   With full fine-tuning you store five full model copies. With LoRA rank=8 you store
   one base model plus five tiny adapters. What does this mean for storage and latency?
3. The original LoRA paper (Hu et al. 2021) found that rank=4 often matches or beats
   full fine-tuning on language tasks. Why might lower rank sometimes generalise better?

Discuss with your neighbour for 3 minutes. We will debrief as a group.


In [ ]:
# Fine-tune the LoRA-adapted model on MNIST (different domain = transfer scenario).
# Only the LoRA A and B matrices receive gradient updates.

mnist_train = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
mnist_test  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
mnist_train_loader = DataLoader(mnist_train, batch_size=128, shuffle=True)
mnist_test_loader  = DataLoader(mnist_test,  batch_size=128, shuffle=False)

FINETUNE_EPOCHS = 3

for epoch in range(FINETUNE_EPOCHS):
    lora_model.train()
    total_loss = 0
    for images, labels in tqdm(mnist_train_loader, desc=f"LoRA FT {epoch+1}/{FINETUNE_EPOCHS}", leave=False):
        images, labels = images.to(device), labels.to(device)
        lora_optimizer.zero_grad()
        loss = criterion(lora_model(images), labels)
        loss.backward()
        lora_optimizer.step()
        total_loss += loss.item()
    print(f"  Epoch {epoch+1}: avg loss = {total_loss / len(mnist_train_loader):.4f}")

# Evaluate
lora_model.eval()
correct, total_samples = 0, 0
with torch.no_grad():
    for images, labels in mnist_test_loader:
        images, labels = images.to(device), labels.to(device)
        preds = lora_model(images).argmax(dim=1)
        correct       += (preds == labels).sum().item()
        total_samples += labels.size(0)

accuracy = 100 * correct / total_samples
print(f"\nMNIST test accuracy after LoRA fine-tuning: {accuracy:.2f}%")
print("(Accuracy is modest because FashionMNIST and MNIST are different domains,")
print(" but the key result is that ONLY the LoRA weights moved -- not the backbone.)")


## Lab 2: Apply LoRA to a New Layer Configuration (Tier 1 Guided, 15 min)

### Situation

The Barclays ML team wants to experiment with different LoRA configurations.
They ask you to build a `replace_fc_with_lora_student` function that applies
LoRA to ALL six fc layers (not just the first three) and supports a configurable rank.

### Task

Complete `replace_fc_with_lora_student` below. It must:
1. Wrap fc1 through fc6 with LoraLayerStudent (your implementation from Lab 1).
2. Accept `rank` and `lora_alpha` as arguments.
3. Return the modified model.

After applying, verify the trainable parameter count printed by the verification cell
matches the expected value (approximately 2x the Lab 1 count because you wrap 6 layers
instead of 3).

### Action

Fill in the function body following the numbered step comments.

### Result

Verification cell prints: "Trainable parameters: X" and "Lab 2 PASSED."


In [ ]:
def replace_fc_with_lora_student(model: FFNModel, rank: int = 8, lora_alpha: int = 16) -> FFNModel:
    """
    Wrap all six fc layers with LoraLayerStudent.
    """
    # Step 1-6: replace each fc layer.
    model.fc1 = LoraLayerStudent(model.fc1, rank=rank, lora_alpha=lora_alpha)
    model.fc2 = LoraLayerStudent(model.fc2, rank=rank, lora_alpha=lora_alpha)
    model.fc3 = LoraLayerStudent(model.fc3, rank=rank, lora_alpha=lora_alpha)
    model.fc4 = LoraLayerStudent(model.fc4, rank=rank, lora_alpha=lora_alpha)
    model.fc5 = LoraLayerStudent(model.fc5, rank=rank, lora_alpha=lora_alpha)
    model.fc6 = LoraLayerStudent(model.fc6, rank=rank, lora_alpha=lora_alpha)

    # Step 7: return the modified model.
    return model


In [ ]:
# Safety-net: run this if you did not finish Lab 2.
# SKIP this cell if you DID finish Lab 2.
import copy as _copy
_test_model = FFNModel()
_needs_safety_2 = False
try:
    _patched = replace_fc_with_lora_student(_copy.deepcopy(_test_model), rank=4)
    if _patched.fc6 is None or not isinstance(_patched.fc6, (LoraLayer, LoraLayerStudent)):
        _needs_safety_2 = True
except Exception:
    _needs_safety_2 = True

if _needs_safety_2:
    print("Using Lab 2 safety-net so the rest of the notebook can run.")
    def replace_fc_with_lora_student(model, rank=8, lora_alpha=16):
        model.fc1 = LoraLayerStudent(model.fc1, rank=rank, lora_alpha=lora_alpha)
        model.fc2 = LoraLayerStudent(model.fc2, rank=rank, lora_alpha=lora_alpha)
        model.fc3 = LoraLayerStudent(model.fc3, rank=rank, lora_alpha=lora_alpha)
        model.fc4 = LoraLayerStudent(model.fc4, rank=rank, lora_alpha=lora_alpha)
        model.fc5 = LoraLayerStudent(model.fc5, rank=rank, lora_alpha=lora_alpha)
        model.fc6 = LoraLayerStudent(model.fc6, rank=rank, lora_alpha=lora_alpha)
        return model
else:
    print("Lab 2 complete: using your replace_fc_with_lora_student.")


In [ ]:
# Verification: do not edit this cell.
import copy

lab2_model = replace_fc_with_lora_student(copy.deepcopy(FFNModel()), rank=8, lora_alpha=16)
lab2_trainable = sum(p.numel() for p in lab2_model.parameters() if p.requires_grad)

assert isinstance(lab2_model.fc6, (LoraLayer, LoraLayerStudent)), \
    "fc6 must be wrapped with LoraLayer (or LoraLayerStudent)"
assert lab2_trainable > trainable_lora, \
    "Wrapping 6 layers must yield more trainable params than wrapping 3"

print(f"Trainable parameters (6 layers wrapped): {lab2_trainable:,}")
print(f"Trainable parameters (3 layers wrapped): {trainable_lora:,}")
print(f"Ratio: {lab2_trainable / trainable_lora:.2f}x (should be approximately 2x)")
print()
print("Lab 2 PASSED.")


### Stretch (fast finishers)

Implement the weight merge for the fully-wrapped model.
Call `merge_weights()` on each LoraLayer after fine-tuning completes.
Verify that:
- Forward pass output is identical before and after merging (difference < 1e-5).
- The merged model has zero trainable parameters (all weights are now baked in).

### Homework Extension

Run the six-layer LoRA model with rank=1, rank=4, rank=8, rank=16.
For each rank: record trainable parameter count and MNIST test accuracy after 3 epochs.
Plot accuracy vs rank on a log-scale x-axis.
At what rank does the accuracy plateau? Does this match the heuristic from the paper
(Hu et al. found rank=4 often sufficient for language tasks)?


In [ ]:
# Homework Extension: sweep over ranks and record param counts.
import copy as _copy_hw

ranks_to_try = [1, 4, 8, 16]
results = []
for r in ranks_to_try:
    m = replace_fc_with_lora_student(_copy_hw.deepcopy(FFNModel()), rank=r, lora_alpha=2 * r)
    n_train = sum(p.numel() for p in m.parameters() if p.requires_grad)
    results.append((r, n_train))
    print(f"rank={r:>3}: trainable params = {n_train:,}")

print()
print("Plot accuracy vs rank after running 3 epochs of MNIST fine-tuning per rank.")
print("Expected pattern: accuracy plateaus around r=4 to r=8 for this small task.")


## Section 3: Rank and Alpha: How to Choose

You have now used rank=8. But how do practitioners choose rank in production?

### Rank heuristics (from Hu et al. 2021 and community practice):

| Rank | Typical use case                                          |
|------|-----------------------------------------------------------|
| 1-2  | Extremely constrained memory; usually too little capacity |
| 4    | Simple tasks, domain adaptation, sufficient for most NLP  |
| 8    | Default starting point; good balance of capacity vs cost  |
| 16   | Complex tasks, multi-domain; diminishing returns above 16 |
| 32+  | Rarely needed; often overfits on small datasets           |

Key principle: start with r=8, monitor validation loss, increase to r=16 only if
underfitting, decrease to r=4 if overfitting. Never set r greater or equal to min(d, k).

### Alpha heuristics:

- alpha = rank: neutral scaling (scale = 1.0)
- alpha = 2 * rank: common default; slightly amplifies the LoRA update
- alpha = rank / 2: conservative; use when base model is already close to target task

The PEFT library default is alpha = lora_r (1:1 ratio). Most practitioners set
alpha = 2 * rank as a starting point because it stabilises training on small datasets.


In [ ]:
# Visualise how rank affects trainable parameter count for Flan-T5-small.

import matplotlib.pyplot as plt

# Flan-T5-small approximate q and v projection sizes
n_projection_pairs = 64
d_size             = 512
k_size             = 512
total_base_params  = 77_000_000

ranks = [1, 2, 4, 8, 16, 32, 64]
lora_params = [n_projection_pairs * (d_size * r + r * k_size) for r in ranks]
fractions   = [100 * p / total_base_params for p in lora_params]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar([str(r) for r in ranks], lora_params, color="steelblue")
axes[0].set_xlabel("LoRA Rank r")
axes[0].set_ylabel("Trainable Parameters")
axes[0].set_title("Trainable Parameters vs Rank (Flan-T5-small, q+v)")
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e3:.0f}K"))
for i, (r, p) in enumerate(zip(ranks, lora_params)):
    axes[0].text(i, p + 500, f"{p/1e3:.0f}K", ha="center", fontsize=8)

axes[1].bar([str(r) for r in ranks], fractions, color="coral")
axes[1].set_xlabel("LoRA Rank r")
axes[1].set_ylabel("% of Total Parameters Trainable")
axes[1].set_title("Trainable Fraction vs Rank (Flan-T5-small)")
for i, (r, f) in enumerate(zip(ranks, fractions)):
    axes[1].text(i, f + 0.002, f"{f:.3f}%", ha="center", fontsize=8)

plt.tight_layout()
plt.savefig("lora_rank_comparison.png", dpi=100, bbox_inches="tight")
plt.show()

print("\nAt rank=8: trainable params =", f"{lora_params[ranks.index(8)]:,}",
      f"({fractions[ranks.index(8)]:.3f}% of total)")


### Peer Discussion (3 min): LoRA in Production

Your Barclays platform serves 10 million customer contacts per month across
complaint classification, summarization, NER, risk flagging, and translation.

1. Full fine-tuning: 5 tasks * 250MB (Flan-T5-small) = 1.25 GB of model files.
   LoRA rank=8: 5 adapters * ~2 MB + 1 frozen base (250 MB) = ~260 MB total.
   Where does this saving matter most: storage, serving latency, or update frequency?

2. When you update the complaint summarization adapter (say, to handle new terminology),
   how does the deployment pipeline differ between full fine-tuning and LoRA?

3. What are the risks of setting alpha too high? Too low?

Discuss for 3 minutes, then we move to the capstone.


## Section 4: Capstone: LoRA Fine-Tune Flan-T5 on SageMaker

You have implemented LoRA from scratch and understood the math.
Now we use the HuggingFace PEFT library to do the same thing on Flan-T5-small,
a real sequence-to-sequence LLM, and run the training job on a GPU instance.

The PEFT library's LoraConfig does exactly what your LoraLayer does:
- Freezes the base model weights.
- Injects low-rank A and B matrices into target modules (q and v projections).
- Saves only the adapter weights (a few MB) rather than the full model.

Training runs on ml.g4dn.xlarge (NVIDIA T4, 16 GB VRAM) and completes in ~10 minutes.
While it runs you will inspect the CloudWatch logs and parameter counts.


In [ ]:
# Before launching the job, let's see which modules PEFT will target.
# This is how you discover target_modules for any model family.

from transformers import AutoModelForSeq2SeqLM

inspect_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

print("All linear layers in Flan-T5-small (showing name and shape):")
print("-" * 65)
lm_linear_count = 0
for name, module in inspect_model.named_modules():
    if isinstance(module, nn.Linear):
        lm_linear_count += 1
        print(f"  {name:<55} {list(module.weight.shape)}")
print("-" * 65)
print(f"Total linear layers: {lm_linear_count}")
print()
print("PEFT target_modules=['q', 'v'] matches any layer whose name ends with 'q' or 'v'.")
print("This covers self-attention and cross-attention projections in encoder and decoder.")

total_flan = sum(p.numel() for p in inspect_model.parameters())
print(f"\nFlan-T5-small total parameters: {total_flan:,}")

del inspect_model


In [ ]:
import os
import boto3

# scripts_topic7a/ must contain exactly:
#   train.py          -- the training entry point
#   requirements.txt  -- auto-installed by SageMaker toolkit

SOURCE_DIR = "scripts_topic7a"

required_files = ["train.py", "requirements.txt"]
for fname in required_files:
    fpath = os.path.join(SOURCE_DIR, fname)
    assert os.path.exists(fpath), f"Missing required file: {fpath}"
    print(f"  Found: {fpath}")

print(f"\nSource dir '{SOURCE_DIR}' is ready.")
print("The HuggingFace estimator will upload this directory to S3 automatically.")


In [ ]:
from sagemaker.huggingface import HuggingFace

# HuggingFace estimator -- GPU ONLY. Never use ml.m5 with this estimator.
# transformers_version="4.56.2", pytorch_version="2.8.0", py_version="py312".
# requirements.txt installs peft>=0.6.0 into the container.

estimator = HuggingFace(
    entry_point="train.py",
    source_dir=SOURCE_DIR,
    role=role,

    transformers_version="4.56.2",
    pytorch_version="2.8.0",
    py_version="py312",

    instance_type="ml.g4dn.xlarge",
    instance_count=1,

    hyperparameters={
        "rank":       8,
        "alpha":      16,
        "epochs":     3,
        "batch_size": 8,
        "lr":         3e-4,
    },

    base_job_name="lora-flan-t5",
    output_path=f"s3://{bucket}/lora-flan-t5/output",
)

print("Estimator defined:")
print(f"  instance_type:        {estimator.instance_type}")
print(f"  transformers_version: {estimator.transformers_version}")
print(f"  pytorch_version:      {estimator.pytorch_version}")
print(f"  hyperparameters:      {estimator.hyperparameters}")


In [ ]:
# Launch the training job asynchronously (wait=False).
# The job runs on ml.g4dn.xlarge; expected time ~10 minutes.

estimator.fit(wait=False)

training_job_name = estimator.latest_training_job.name
print(f"Training job launched: {training_job_name}")
print()
print("Monitor in AWS Console:")
print(f"  SageMaker > Training > Training jobs > {training_job_name}")
print()
print("Or run the polling cell below to check status every 60 seconds.")


In [ ]:
# Safety-net: run this if your kernel restarted after launching the training job.
# SKIP this cell if training_job_name is already defined.
if 'training_job_name' not in dir() or training_job_name is None:
    training_job_name = "<PASTE YOUR JOB NAME HERE>"
    print(f"Using safety-net training_job_name: {training_job_name}")
else:
    print(f"training_job_name already defined: {training_job_name}")


In [ ]:
import time

sm_client = boto3.client("sagemaker", region_name=region)

print(f"Polling job: {training_job_name}")
print("-" * 50)

while True:
    response = sm_client.describe_training_job(TrainingJobName=training_job_name)
    status   = response["TrainingJobStatus"]
    elapsed  = response.get("TrainingTimeInSeconds", 0)
    print(f"  Status: {status:<12} | Elapsed: {elapsed}s")

    if status in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(60)

if status == "Completed":
    print("\nTraining job COMPLETED successfully.")
    model_s3_uri = response["ModelArtifacts"]["S3ModelArtifacts"]
    print(f"Adapter artifacts at: {model_s3_uri}")
else:
    print(f"\nTraining job ended with status: {status}")
    failure_reason = response.get("FailureReason", "No reason provided")
    print(f"Failure reason: {failure_reason}")


In [ ]:
# Stream the last 50 log lines from CloudWatch.

import boto3

logs_client = boto3.client("logs", region_name=region)
log_group   = "/aws/sagemaker/TrainingJobs"

try:
    streams = logs_client.describe_log_streams(
        logGroupName=log_group,
        logStreamNamePrefix=training_job_name,
        orderBy="LastEventTime",
        descending=True,
        limit=1,
    )["logStreams"]

    if streams:
        stream_name = streams[0]["logStreamName"]
        events = logs_client.get_log_events(
            logGroupName=log_group,
            logStreamName=stream_name,
            limit=50,
            startFromHead=False,
        )["events"]
        print(f"Last 50 log lines from {stream_name}:\n")
        for event in events:
            print(" ", event["message"])
    else:
        print("No log streams found yet. Wait a moment and re-run this cell.")
except Exception as e:
    print(f"Could not retrieve logs: {e}")
    print("You can view logs in the AWS Console under CloudWatch > Log Groups.")


## What the Training Logs Tell You

Look for these lines in the CloudWatch output:

1. "Base model parameters: 77,000,000" : Flan-T5-small total
2. "Trainable parameters with LoRA: ~300,000" : only A and B matrices
3. "Trainable fraction: 0.39%" : you are training less than half a percent
4. Epoch 1-3 loss values : should decrease steadily
5. "final_rouge1: 0.XX" : token overlap metric for summarization quality

The adapter checkpoint saved to S3 is typically 2-5 MB, not 250 MB.
You can load it back with:

    from peft import PeftModel
    base = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")
    model = PeftModel.from_pretrained(base, "<path to adapter>")

This is the production pattern: one frozen base, many tiny adapters.


In [ ]:
# Optional: run this cell after training completes to test the adapter locally.

import os, json

local_adapter_path = "./lora_adapter_local"

if os.path.exists(local_adapter_path):
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    from peft import PeftModel

    tokenizer   = AutoTokenizer.from_pretrained("google/flan-t5-small")
    base_model  = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")
    peft_model  = PeftModel.from_pretrained(base_model, local_adapter_path)
    peft_model.eval()

    test_complaints = [
        "My credit card was charged twice and nobody refunded me after three calls.",
        "The mobile app crashes whenever I try to check my savings balance.",
    ]

    print("Complaint Summarization (LoRA-adapted Flan-T5-small):")
    print("-" * 60)
    for complaint in test_complaints:
        inputs = tokenizer(
            "summarize: " + complaint,
            return_tensors="pt", max_length=256, truncation=True)
        with torch.no_grad():
            outputs = peft_model.generate(**inputs, max_new_tokens=32)
        summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(f"Input:   {complaint[:70]}...")
        print(f"Summary: {summary}")
        print()
else:
    print("Adapter not found locally. Download from S3 first:")
    print(f"  aws s3 cp <model_s3_uri> {local_adapter_path}/ --recursive")


## Wrap-Up: What You Built in Topic 7a

### Key takeaways

1. Full fine-tuning stores delta_W as a dense matrix: the same size as W itself.
   For a 250M parameter model that is ~950 MB of additional weights per task.

2. LoRA factorises delta_W = B @ A where r is much smaller than min(d, k).
   At rank=8 on Flan-T5-small: 0.39% of parameters are trainable.
   Adapters are 2-5 MB, not 250 MB.

3. Initialisation matters: B = zeros ensures the adapted model starts identical
   to the pre-trained model. Only during training do A and B diverge.

4. Rank selection heuristics: start at r=8. Increase to r=16 if underfitting.
   Decrease to r=4 if overfitting. r=1-2 rarely works for language tasks.

5. The PEFT library (get_peft_model, LoraConfig) automates exactly what you built
   in LoraLayer: freeze base, inject A and B, save only adapters.

### How this connects to Topic 7b

Topic 7b applies PEFT LoRA to DistilBERT for a classification task (encoder-only model).
You will see how target_modules changes for an encoder architecture, and how to combine
LoRA with 8-bit quantisation (QLoRA) to fit even larger models on the same T4 GPU.

### The Barclays system so far

Recent topics have added:
- Topic 6a: Full fine-tuning Flan-T5 (all params updated, catastrophic forgetting demo)
- Topic 6b: Transfer learning DistilBERT (frozen backbone, head only)
- Topic 7a: LoRA on FFN + Flan-T5 (freeze everything, two matrices per layer)
- Next: Topic 7b: PEFT and LoRA with DistilBERT


In [ ]:
# Summary of what was built and trained in this topic.

print("Topic 7a Summary")
print("=" * 55)
print()
print("LoRA implementation:")
print(f"  LoraLayer class: DONE")
print(f"  FFN with LoRA adapters: DONE")
print(f"  Trainable params (FFN, 3 layers, r=8): {trainable_lora:,}")
print(f"  Compression vs full FT: {train_pre / trainable_lora:.1f}x")
print()
print("SageMaker capstone:")
print(f"  Model: google/flan-t5-small (77M params)")
print(f"  LoRA rank: 8, alpha: 16, target: q + v projections")
print(f"  Job: {training_job_name}")
print(f"  Instance: ml.g4dn.xlarge (NVIDIA T4)")
print()
print("Next: Topic 7b: PEFT LoRA on DistilBERT (encoder-only, classification)")
